# Distributed File Processing with TaskVine

This is a starter notebook for a Floability backpack. It demonstrates:
1. Connecting to a TaskVine manager
2. Defining a distributed worker function
3. Processing files from the `data/` directory

**To customize this notebook:**
- Edit the `process_file()` function to your needs
- Modify the file list or directory to match your data
- Add more worker functions or distribute more tasks

## Setup: Manager Connection

TaskVine requires a manager process. The manager name and ports come from environment variables set by Floability.

In [ ]:
import os
import ndcctools.taskvine as vine

def parse_ports(ports_str: str) -> list[int]:
    ports = [int(p.strip()) for p in ports_str.split(",") if p.strip()]

    if not ports:
        raise ValueError("No valid ports provided")

    return ports

name = os.environ.get("VINE_MANAGER_NAME")

# Get manager info from environment (set by Floability)
manager_name = os.environ.get('VINE_MANAGER_NAME')
manager_ports = parse_ports(os.environ.get("VINE_MANAGER_PORTS", "9123,9150"))

print(f'Manager Name: {manager_name}')
print(f'Manager Ports: {manager_ports}')

m = vine.Manager(manager_ports, name=manager_name)
print(f"[manager] Listening on port {m.port}")

## Define Worker Function

The worker function will run on distributed worker nodes. It reads a file and returns its byte size.

**To customize:** Replace the function body with your processing logic.

In [ ]:
def process_file(file_path):
    """Process a file and return its byte size.
    
    Args:
        file_path: Path to the file to process
    
    Returns:
        Dictionary with file metadata
    """
    import os
    
    if not os.path.exists(file_path):
        return {'file': file_path, 'error': 'File not found'}
    
    file_size = os.path.getsize(file_path)
    
    return {
        'file': file_path,
        'size_bytes': file_size,
    }

# Test the function locally
print('Worker function defined.')

## Submit Tasks to Workers

This cell submits file processing tasks to the TaskVine workers.

**To customize:** Modify the file list to match your `data/` directory contents.

In [ ]:
import os
import glob

# List files in the data directory
data_dir = 'data'
if os.path.isdir(data_dir):
    files = glob.glob(os.path.join(data_dir, '*'))
else:
    files = []
    print(f'Warning: {data_dir} directory not found')

print(f'Found {len(files)} file(s) in {data_dir}/')
for f in files[:5]:  # Show first 5
    print(f'  - {f}')

# Submit tasks to manager
# (Requires TaskVine manager connection from above)
if 'q' in locals():
    for file_path in files:
        # Create task with the worker function
        t = vine.PythonTask(
            process_file,
            file_path,
        )
        q.submit(t)
    
    print(f'Submitted {len(files)} task(s) to the manager')
else:
    print('Manager not connected. Cannot submit tasks.')

## Collect Results

Wait for tasks to complete and collect results.

In [ ]:
if 'q' in locals():
    results = []
    while len(results) < q.hungry():
        t = q.wait(5)
        if t:
            result = t.output
            results.append(result)
            print(f'Task completed: {result}')
    
    print(f'\nAll {len(results)} task(s) completed!')
    # Display summary
    total_size = sum(r.get('size_bytes', 0) for r in results if 'size_bytes' in r)
    print(f'Total size processed: {total_size} bytes')
else:
    print('Manager not connected.')